In [1]:
# import basic stuff
import sklearn
import pandas as pd
import numpy as np
import itertools
from tqdm import tqdm

# import tensorflow and keras stuff
!pip install tensorflow_probability
import tensorflow_probability as tfp
import tensorflow as tf

tfd = tfp.distributions
from keras.layers import *
from keras.models import *
from keras.callbacks import *
from keras.optimizers import *
from keras.losses import *
from keras.regularizers import *
import keras.backend as K

# import kflod stuff
from sklearn.model_selection import KFold

# import preprocessing functions
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error

# import comparison models
from xgboost import XGBClassifier, XGBRegressor


!pip install interpret
from interpret.glassbox import (
    ExplainableBoostingClassifier,
    ExplainableBoostingRegressor,
)



# plotting
import matplotlib.pyplot as plt


from scipy.stats import entropy, wasserstein_distance

!pip install properscoring
import properscoring as ps


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 49.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.9/547.9 kB 52.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.0/758.0 kB 61.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 33.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 95.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 124.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 28.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 30.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 18.2 MB/s eta

In [2]:
!pip install nodegam
from nodegam.sklearn import NodeGAMRegressor, NodeGAMClassifier
from nodegam.gams.MySpline import MySplineLogisticGAM, MySplineGAM
from nodegam.gams.MyEBM import MyExplainableBoostingClassifier, MyExplainableBoostingRegressor
from nodegam.gams.MyXGB import MyXGBOnehotClassifier, MyXGBOnehotRegressor
from nodegam.gams.MyBagging import MyBaggingClassifier, MyBaggingRegressor
from nodegam.utils import sigmoid_np, average_GAM_dfs
from nodegam.vis_utils import vis_GAM_effects

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 MB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.2/522.2 kB 26.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 30.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.4
    Uninstalling numpy-1.22.4:
      Successfully uninsta

WARNING! The apex is not installed so fp16 is not available.


In [3]:
def kl_divergence(predicted_mu, predicted_sigma, y_test):
    y_dist = np.expand_dims(y_test, 1)
    y_dist = y_dist.astype(np.float32)


    if predicted_mu.shape != (len(y_test), 1):
        predicted_mu = np.expand_dims(y_test, 1)
    try:
        predicted_mu = predicted_mu.astype(np.float32)
    except:
        pass

    try:
        predicted_sigma = predicted_sigma.astype(np.float32)
    except:
        pass

    t = tfd.Normal(loc=y_dist, scale=np.std(y_dist))

    p = tfd.Normal(loc=predicted_mu, scale=predicted_sigma)


    kl = tf.reduce_mean(tfd.kl_divergence(t, p, allow_nan_stats=True))

    return kl.numpy()


def stable_kl_div(predicted_mu, predicted_sigma, y_test):
    y_dist = np.expand_dims(y_test, 1)
    y_dist = y_dist.astype(np.float32)


    if predicted_mu.shape != (len(y_test), 1):
        predicted_mu = np.expand_dims(y_test, 1)
    try:
        predicted_mu = predicted_mu.astype(np.float32)
    except:
        pass

    try:
        predicted_sigma = predicted_sigma.astype(np.float32)
    except:
        pass

    t = tfd.Normal(loc=y_dist, scale=np.std(y_dist)).log_prob(y_dist)

    p = tfd.Normal(loc=predicted_mu, scale=predicted_sigma).log_prob(predicted_mu)

    # Calculate the mean KL divergence using the log PDFs
    kl_divergence = tf.reduce_mean(p - t)

    return kl_divergence.numpy()


def compute_wasserstein_distance(p, q):
    return wasserstein_distance(p, q)

from scipy.stats import norm


def crps(predicted_mean, predicted_std, true_value):
    """
    Calculate the Continuous Ranked Probability Score (CRPS) for a normal distribution.

    Parameters:
    predicted_mean (float or numpy array): Predicted mean.
    predicted_std (float or numpy array): Predicted standard deviation.
    true_value (float or numpy array): True value(s).

    Returns:
    float: CRPS score.
    """
    cdf_diff = (norm.cdf(true_value, loc=predicted_mean, scale=predicted_std) -
                np.where(true_value >= predicted_mean, 1, 0))
    crps = np.mean(cdf_diff ** 2)
    return crps

def crps_norm(y_true, mu, sigma=None):
    if sigma is None:
      crps = ps.crps_ensemble(y_true, mu).mean()
    else:
      crps = ps.crps_gaussian(y_true, mu, sigma).mean()

    return crps

In [4]:

####################################### NAM EXU-activation Layer
class ExuLayer(tf.keras.layers.Layer):
    def __init__(self, units=32, input_dim=32):
        super(ExuLayer, self).__init__()
        w_init = tf.random_normal_initializer()
        self.w = tf.Variable(
            initial_value=w_init(shape=(input_dim, units), dtype="float32"),
            trainable=True,
        )
        b_init = tf.zeros_initializer()
        self.b = tf.Variable(
            initial_value=b_init(shape=(units,), dtype="float32"), trainable=True
        )

    def call(self, inputs):
        return tf.clip_by_value(tf.matmul(inputs, tf.exp(self.w)) + self.b, 0, 1)

In [5]:
class CustomPipeline(Pipeline):
    """Custom sklearn Pipeline to transform data."""

    def apply_transformation(self, x):
        """Applies all transforms to the data, without applying last estimator.

        Args:
          x: Iterable data to predict on. Must fulfill input requirements of first
            step of the pipeline.

        Returns:
          xt: Transformed data.
        """
        xt = x
        for _, transform in self.steps[:-1]:
            xt = transform.fit_transform(xt)
        return xt


def transform_data(df):
    """Apply a fixed set of transformations to the pd.Dataframe `df`.

    Args:
      df: Input dataframe containing features.

    Returns:
      Transformed dataframe and corresponding column names. The transformations
      include (1) encoding categorical features as a one-hot numeric array, (2)
      identity `FunctionTransformer` for numerical variables. This is followed by
      scaling all features to the range (-1, 1) using min-max scaling.
    """
    column_names = df.columns
    new_column_names = []
    is_categorical = np.array([dt.kind == "O" for dt in df.dtypes])
    categorical_cols = df.columns.values[is_categorical]
    numerical_cols = df.columns.values[~is_categorical]
    for index, is_cat in enumerate(is_categorical):
        col_name = column_names[index]
        if is_cat:
            new_column_names += [
                "{}: {}".format(col_name, val) for val in set(df[col_name])
            ]
        else:
            new_column_names.append(col_name)
    cat_ohe_step = ("ohe", OneHotEncoder(sparse=False, handle_unknown="ignore"))

    cat_pipe = Pipeline([cat_ohe_step])
    num_pipe = Pipeline([("identity", FunctionTransformer(validate=True))])
    transformers = [
        ("cat", cat_pipe, categorical_cols),
        ("num", num_pipe, numerical_cols),
    ]
    column_transform = ColumnTransformer(transformers=transformers)

    pipe = CustomPipeline(
        [
            ("column_transform", column_transform),
            ("min_max", MinMaxScaler((-1, 1))),
            ("dummy", None),
        ]
    )
    df = pipe.apply_transformation(df)
    return df, new_column_names


In [6]:



######################################################### Model builder


############################### Helper functions for building MLP, NAM and NAMLSS
def built_DNN(input, output_activation="linear", output_num=1):
    x = Dense(1000, "relu")(input)
    x = Dropout(0.5)(x)
    x = Dense(500, "relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(50, "relu")(x)
    x = Dense(25, "relu")(x)
    #x = Dense(128, "relu")(input)
    #x = Dropout(0.3)(x)
    #x = Dense(64, "relu")(x)
    #x = Dropout(0.3)(x)
    #x = Dense(32, "relu")(x)
    x = Dense(output_num, activation=output_activation, use_bias=False)(x)
    model_dnn = Model(inputs=input, outputs=x)
    model_dnn.reset_states()
    return model_dnn


def LINEAR(x):
    return x


################# MLP
def MLP(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    output_activation="linear",
):
    inps = Input(shape=(features_train.shape[1],))
    model = built_DNN(inps, output_activation=output_activation)

    model.compile(
        loss=POINT_LOSS, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    history = model.fit(
        x=features_train,
        y=labels_train,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    loc_pred = model.predict(features_test)
    loc_pred = np.array([loc_pred[i][0] for i in range(len(loc_pred))], dtype=np.float64)
    likelihood = LL_EVAL(loc_pred, labels_test)

    ll, point_loss, kl, skl, ws, crsp = LL_EVAL(loc_pred, labels_test)

    return ll, point_loss, kl, skl, ws, crsp


######################## Distributional DNN


def DDNN(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    distribution=tfd.Normal,
    loc_activation=LINEAR,
    scale_activation=tf.math.softplus,
    output_num=2,
):
    # Create inputs
    inps = Input(shape=(features_train.shape[1],))
    ms = built_DNN(inps, "linear", output_num)
    z = ms.output

    # built distributional layer
    # Change for when dist params have different names
    p_y = tfp.layers.DistributionLambda(
        lambda x: distribution(
            loc=loc_activation(x[:, 0]), scale=scale_activation(x[:, 1])
        )
    )(z)

    model = Model(inputs=ms.input, outputs=p_y)

    def NLL(y_true, y_hat):
        return -y_hat.log_prob(y_true)

    model.compile(
        loss=NLL, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    history = model.fit(
        x=features_train,
        y=labels_train,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    # Evaluate model
    preds = ms(features_test)
    mu_preds = np.array(
        loc_activation(preds[:, 0])
    )
    sigma_preds = np.array(
        [
            scale_activation(preds[:, 1])
        ]
    )

    ll, point_loss, kl, skl, ws, crsp = LL_EVAL(mu_preds, labels_test, sigma_preds)

    return ll, point_loss, kl, skl, ws, crsp


######################################### NAM


def NAM(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    output_activation="linear",
):
    inps = [Input(shape=(1,)) for _ in range(features_train.shape[1])]

    # define submodels
    # same architecture as for DNN and MLP
    ms = [
        built_DNN(inps[i], output_activation=output_activation)
        for i in range(features_train.shape[1])
    ]
    z = sum([m.output for m in ms])
    model = Model(inputs=[m.input for m in ms], outputs=z)

    model.compile(
        loss=POINT_LOSS, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    training_data = [features_train[:,i] for i in range(features_train.shape[1])]
    eval_data = [features_test[:,i] for i in range(features_test.shape[1])]

    history = model.fit(
        x=training_data,
        y=labels_train,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    loc_pred = model.predict(eval_data)
    loc_pred = np.array([loc_pred[i][0] for i in range(len(loc_pred))], dtype=np.float64)
    ll, point_loss, kl, skl, ws, crsp = LL_EVAL(loc_pred, labels_test)

    return ll, point_loss, kl, skl, ws, crsp


############################################ NAMLSS


def define_models_scale(input):
    x = Dense(50, activation="relu")(input)
    x = Dense(25, activation="relu")(x)
    x = Dense(1, activation="linear", use_bias=False)(x)
    x = Model(inputs=input, outputs=x)
    # x.reset_states()
    return x


def NAMLSS(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    distribution=tfd.Normal,
    loc_activation=LINEAR,
    scale_activation=tf.math.softplus,
    output_num=1,
):
    training_data = 2 * [features_train[:, i] for i in range(features_train.shape[1])]
    eval_data = 2 * [features_test[:, i] for i in range(features_test.shape[1])]

    inps = [Input(shape=(1,)) for _ in range(2 * features_train.shape[1])]

    ms = [built_DNN(inps[i]) for i in range(features_train.shape[1])]
    ms += [
        define_models_scale(inps[i + features_train.shape[1]])
        for i in range(features_train.shape[1])
    ]

    z1 = sum([m.output for m in ms[: features_train.shape[1]]])
    z2 = sum([m.output for m in ms[features_train.shape[1] :]])

    z = concatenate([z1, z2])

    # Change for when dist params have different names
    p_y = tfp.layers.DistributionLambda(
        lambda x: distribution(
            loc=loc_activation(x[:, 0]), scale=scale_activation(x[:, 1])
        )
    )(z)

    model = Model(inputs=[m.input for m in ms], outputs=p_y)

    def NLL(y_true, y_hat):
        return -y_hat.log_prob(y_true)

    model.compile(
        loss=NLL, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    history = model.fit(
        x=training_data,
        y=labels_train,
        validation_split=0.2,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    preds = [ms[idx].predict(eval_data[idx], verbose=0) for idx in range(len(eval_data))]
    preds_mu = preds[:features_test.shape[1]]
    preds_sigma = preds[features_test.shape[1]:]

    mu = sum(preds_mu)
    sigma = sum(preds_sigma)
    mu_preds = loc_activation(mu)
    sigma_preds = scale_activation(sigma)

    mu = np.array([mu_preds[i][0] for i in range(len(mu_preds))])
    sigma = np.array([sigma_preds[i][0] for i in range(len(sigma_preds))])

    ll, point_loss, kl, skl, ws, crsp = LL_EVAL(mu, labels_test, sigma)

    return ll, point_loss, kl, skl, ws, crsp


#### NAMLSS 2
def NA2MLSS(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    distribution=tfd.Normal,
    loc_activation=LINEAR,
    scale_activation=tf.math.softplus,
    output_num=2,
):
    training_data = [features_train[:, i] for i in range(features_train.shape[1])]
    eval_data = [features_test[:, i] for i in range(features_test.shape[1])]

    inps = [Input(shape=(1,)) for _ in range(features_train.shape[1])]

    ms = [
        built_DNN(inps[i], output_num=output_num)
        for i in range(features_train.shape[1])
    ]

    z = sum([m.output for m in ms])

    # Change for when dist params have different names
    p_y = tfp.layers.DistributionLambda(
        lambda x: distribution(
            loc=loc_activation(x[:, 0]), scale=scale_activation(x[:, 1])
        )
    )(z)

    model = Model(inputs=[m.input for m in ms], outputs=p_y)

    def NLL(y_true, y_hat):
        return -y_hat.log_prob(y_true)

    model.compile(
        loss=NLL, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    history = model.fit(
        x=training_data,
        y=labels_train,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    preds = [ms[idx].predict(eval_data[idx], verbose=0) for idx in range(len(eval_data))]

    preds = sum(preds)
    mu, sigma = preds[:,0], preds[:, 1]
    mu_preds = loc_activation(mu)
    sigma_preds = scale_activation(sigma)

    ll, point_loss, kl, skl, ws, crsp = LL_EVAL(mu_preds, labels_test, sigma_preds)

    return ll, point_loss, kl, skl, ws, crsp


###################### XGBoost
def XGB(features_train, labels_train, features_test, labels_test, regression=True):
    if regression:
        model = XGBRegressor()
    else:
        model = XGBClassifier()
    model.fit(features_train, labels_train)

    preds = model.predict(features_test)

    ll, point_loss, kl, skl, ws, crsp = LL_EVAL(np.float64(preds), labels_test)

    return ll, point_loss, kl, skl, ws, crsp




############## EBM
def EBM(features_train, labels_train, features_test, labels_test, regression=True):
    if regression:
        model = ExplainableBoostingRegressor()
    else:
        model = ExplainableBoostingClassifier()

    model.fit(features_train, labels_train)

    preds = model.predict(features_test)

    ll, point_loss, kl, skl, ws, crsp = LL_EVAL(np.float64(preds), labels_test)

    return ll, point_loss, kl, skl, ws, crsp


############################## NODEGAM
def NODEGAM(features_train, labels_train, features_test, labels_test, regression=True):
    if regression:
        model = NodeGAMRegressor(
            in_features=features_train.shape[1], verbose=0, seed=141, ga2m=0
        )
    else:
        model = NodeGAMClassifier(
            in_features=features_train.shape[1],
            verbose=0,
            seed=141,
            ga2m=0,
        )

    X_train = pd.DataFrame(np.vstack([features_train])).reset_index(drop=True)
    X_test = pd.DataFrame(np.vstack([features_test])).reset_index(drop=True)
    record = model.fit(X_train, np.array(labels_train))
    preds = model.predict(X_test)

    ll, point_loss, kl, skl, ws, crsp = LL_EVAL(np.float64(preds), labels_test)

    return ll, point_loss, kl, skl, ws, crsp


In [7]:

if __name__ == "__main__":
    # task:
    REGRESSION = True
    # general arguments
    BATCH_SIZE = 1024
    NUM_EPOCHS = 2000
    LEARNING_RATE = 0.001
    NUM_FOLDS = 5

    # loss func for point estimators
    POINT_LOSS = "mse"

    EARLY_STOPPING = EarlyStopping(
        patience=150, restore_best_weights=True, min_delta=1e-05, monitor="loss"
    )

    METRICS = [tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"]

    REDUCE_LR = ReduceLROnPlateau(
        monitor="loss", factor=0.95, patience=25, min_delta=1e-05
    )

    KFOLD = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=101)

    DISTRIBUTION = tfd.Normal

    # Define distribution that is modelled
    def LL_EVAL(loc, y_true, scale=None):
        if scale is None:
          dist = DISTRIBUTION(loc, scale=np.std(y_true))
        else:
          dist = DISTRIBUTION(loc, scale=scale)
        ll = - tf.reduce_sum(dist.log_prob(value=y_true)).numpy()
        point_loss = mean_squared_error(y_true, loc)

        crsp = crps_norm(y_true, loc, scale)
        wasserstein = compute_wasserstein_distance(y_true, loc)
        if scale is None:
          kl_div = kl_divergence(loc, np.std(loc), y_true)
          skl_div = stable_kl_div(loc, np.std(loc), y_true)
        else:
          kl_div = kl_divergence(loc, scale, y_true)
          skl_div = stable_kl_div(loc, scale, y_true)

        print(ll, point_loss, kl_div, skl_div, wasserstein, crsp)

        return ll, point_loss, kl_div, skl_div, wasserstein, crsp


    from sklearn import datasets
    housing = datasets.fetch_california_housing()

    X = pd.DataFrame(data=housing.data, columns=housing.feature_names)

    targets = housing.target
    df = pd.DataFrame(X, columns=housing.feature_names)
    df["targets"] = housing.target

    from scipy import stats
    df = df[(np.abs(stats.zscore(df)) < 10).all(axis=1)]
    df = df.reset_index(drop=True)


    X = df[housing.feature_names]
    targets = np.array(df["targets"])
    # Always use transform_data function on X
    features, cols = transform_data(X)

    scaler = StandardScaler().fit(np.array(targets).reshape(-1, 1))
    targets = scaler.transform(np.array(targets).reshape(-1, 1)).flatten()

    fold_no = 1

    model_list = [
        "MLP",
        "DDNN",
        "NAMLSS",
        "NA2MLSS",
        "NAM",
        "XGBOOST",
        "EBM",
        "NODEGAM",
    ]

    results = pd.DataFrame(columns=["Model", "Likelihood", "MSE", "KL", "SKL", "Wasserstein", "CRSP"])

    for mod in model_list:
        print(mod)
        ll_per_fold = []
        mse_per_fold = []
        kl_div_per_fold = []
        skl_div_per_fold = []
        ws_dis_per_fold = []
        crsp_per_fold = []

        for train, test in tqdm(KFOLD.split(features, targets)):
            if mod == "DDNN":
                ll, pl, kl, skl, ws, crsp = DDNN(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    distribution=DISTRIBUTION,
                    loc_activation=LINEAR,
                    scale_activation=tf.math.softplus,
                    output_num=2,
                )
            elif mod == "MLP":
                ll, pl, kl, skl, ws, crsp = MLP(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    output_activation="linear",
                )

            elif mod == "NAM":
                ll, pl, kl, skl, ws, crsp = NAM(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    output_activation="linear",
                )
            elif mod == "XGBOOST":
                ll, pl, kl, skl, ws, crsp = XGB(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    REGRESSION,
                )
            elif mod == "EBM":
                ll, pl, kl, skl, ws, crsp = EBM(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    REGRESSION,
                )
            elif mod == "NODEGAM":
                ll, pl, kl, skl, ws, crsp = NODEGAM(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    REGRESSION,
                )
            elif mod == "NAMLSS":
                ll, pl, kl, skl, ws, crsp = NAMLSS(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    distribution=DISTRIBUTION,
                    loc_activation=LINEAR,
                    scale_activation=tf.math.softplus,
                    output_num=1,
                )
            elif mod == "NA2MLSS":
                ll, pl, kl, skl, ws, crsp = NA2MLSS(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    distribution=DISTRIBUTION,
                    loc_activation=LINEAR,
                    scale_activation=tf.math.softplus,
                    output_num=2,
                )

            ll_per_fold.append(ll)
            mse_per_fold.append(pl)
            kl_div_per_fold.append(kl)
            skl_div_per_fold.append(skl)
            ws_dis_per_fold.append(ws)
            crsp_per_fold.append(crsp)

        name = NUM_FOLDS*[mod]
        temp_df = pd.DataFrame(np.array((name, ll_per_fold, mse_per_fold, kl_div_per_fold, skl_div_per_fold, ws_dis_per_fold, crsp_per_fold)).T, columns=["Model", "Likelihood", "MSE", "KL", "SKL", "Wasserstein", "CRSP"])
        results = pd.concat([results, temp_df])


MLP


0it [00:00, ?it/s]

129/129 [==============================] - 0s 2ms/step


1it [02:23, 143.63s/it]

4075.105010193082 0.17546168267932588 0.0068698972 0.08065671 0.056812418087406766 0.26569868197236857
4075.105010193082 0.17546168267932588 0.0068698972 0.08065671 0.056812418087406766 0.26569868197236857
129/129 [==============================] - 0s 2ms/step


...

4322.0336856780705 0.25125357662722475 0.030986339 0.16627884 0.14866840070116216 0.35418573820390437
Normalize y. mean = 0.004214591263286725, std = 1.0007367574723751


4it [18:20, 271.82s/it]

4272.744013408477 0.2415263848586197 0.025359944 0.15122485 0.15492750304510414 0.3479792342846783
Normalize y. mean = 0.0035471653887682465, std = 1.000498802604787


5it [21:41, 260.20s/it]

4291.536141844756 0.24908976611518852 0.030176014 0.16420984 0.15461126230500694 0.3539091980771031


In [8]:
results = results.astype({"Likelihood": float})
results = results.astype({"MSE": float})
results = results.astype({"CRSP": float})
results = results.astype({"Wasserstein": float})
results = results.astype({"KL": float})
results = results.astype({"SKL": float})
results.groupby("Model").mean()

,Likelihood,MSE,KL,SKL,Wasserstein,CRSP
Model,,,,,,
DDNN,8433.329000,0.175189,27.492737,1.354904,0.066082,0.193388
EBM,4167.830033,0.186198,0.011145,0.101832,0.095388,0.297105
MLP,4143.314654,0.174314,0.007763,0.085202,0.066563,0.263996
NA2MLSS,2891.896820,0.307356,3.513043,0.934860,0.198360,0.272619
NAM,4315.462695,0.257782,0.027189,0.155026,0.185928,0.369719
NAMLSS,3684.621860,0.343098,5.232384,0.996731,0.207965,0.286763
NODEGAM,4284.636428,0.242822,0.028152,0.158787,0.150105,0.350368
XGBOOST,4128.166931,0.166991,0.006308,0.077332,0.065426,0.270986


In [9]:
results.groupby("Model").std()

,Likelihood,MSE,KL,SKL,Wasserstein,CRSP
Model,,,,,,
DDNN,14671.736424,0.004764,4.204087,0.023955,0.008340,0.002025
EBM,49.466165,0.004737,0.001323,0.005843,0.005228,0.002295
MLP,51.054598,0.005814,0.001686,0.008745,0.010850,0.003710
NA2MLSS,750.026887,0.042653,0.950250,0.080497,0.012448,0.018810
NAM,90.997346,0.038792,0.007558,0.020675,0.073189,0.045098
NAMLSS,408.647542,0.015018,0.363226,0.023760,0.008960,0.005965
NODEGAM,45.454878,0.007515,0.002354,0.006281,0.004608,0.004145
XGBOOST,52.739694,0.004072,0.000456,0.002736,0.005685,0.002030
